# Task 3.1 — Voice Embedding Extraction

In [7]:
!pip install -q speechbrain torchaudio soundfile numpy

In [8]:
import gc
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
import torchaudio

TARGET_SR = 16_000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Load reference audio

In [9]:
def load_mono_16k(path):
    wav_np, sr = sf.read(str(path), always_2d=True, dtype='float32')
    wav = torch.from_numpy(wav_np.T)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav.squeeze(0).contiguous()

ref_path = 'Student_voice_ref.wav'
assert Path(ref_path).exists(), f'Please upload {ref_path}!'
ref_wav = load_mono_16k(ref_path)
print(f'Reference audio: {ref_wav.numel()/TARGET_SR:.1f}s')

Reference audio: 62.2s


## Extract ECAPA-TDNN speaker embedding

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

print('Loading ECAPA-TDNN speaker encoder...')
encoder = EncoderClassifier.from_hparams(
    source='speechbrain/spkrec-ecapa-voxceleb',
    run_opts={'device': str(device)},
)

# Process in chunks for long audio
chunk_s = 10.0
chunk_samples = int(chunk_s * TARGET_SR)
total_samples = ref_wav.numel()

embeddings = []
for start in range(0, total_samples, chunk_samples):
    end = min(total_samples, start + chunk_samples)
    chunk = ref_wav[start:end]
    if chunk.numel() < 3200: 
        continue
    with torch.no_grad():
        emb = encoder.encode_batch(chunk.unsqueeze(0).to(device))
        embeddings.append(emb.squeeze().cpu().numpy())
    gc.collect()

# Average embeddings + L2 normalize
avg_embedding = np.mean(embeddings, axis=0)
avg_embedding = avg_embedding / (np.linalg.norm(avg_embedding) + 1e-12)

print(f'Embedding shape: {avg_embedding.shape}')
print(f'Embedding norm: {np.linalg.norm(avg_embedding):.4f}')

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Loading ECAPA-TDNN speaker encoder...


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Embedding shape: (192,)
Embedding norm: 1.0000


## Validate: cosine similarity with lecture audio

In [ ]:
orig_path = 'original_segment.wav'
if Path(orig_path).exists():
    orig_wav = load_mono_16k(orig_path)
    
    mid = orig_wav.numel() // 2
    sample = orig_wav[max(0,mid-TARGET_SR*15):mid+TARGET_SR*15]
    with torch.no_grad():
        orig_emb = encoder.encode_batch(sample.unsqueeze(0).to(device))
        orig_emb = orig_emb.squeeze().cpu().numpy()
        orig_emb = orig_emb / (np.linalg.norm(orig_emb) + 1e-12)

    cos_sim = np.dot(avg_embedding, orig_emb)
    print(f'Cosine similarity (your voice vs lecture): {cos_sim:.4f}')
    print(f'  (Different speakers should be < 0.3, same speaker > 0.7)')
else:
    print(f'{orig_path} not found, skipping validation')

Cosine similarity (your voice vs lecture): 0.1400
  (Different speakers should be < 0.3, same speaker > 0.7)


## Save embedding

In [12]:
np.save('speaker_embedding.npy', avg_embedding)
print(f'Saved speaker embedding to speaker_embedding.npy')

del encoder
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Done!')

Saved speaker embedding to speaker_embedding.npy
Done!
